#### Imports

In [ ]:
# local imports
from dist_gen import *

# external imports
import matplotlib.animation as anim
import matplotlib.pyplot as plt from collections
import namedtuple
import numpy as np
#import pandas as pd
#import plasmapy as plas
#import random as rd

#### 1D Particle Simulation

Particle Definition and Data Handling

Parameters

In [ ]:
N_e = 1000 # number of electrons
N_i = 1000 # number of ions
x_min, x_max = -10, 10
v_min, v_max = -10, 10
dt = 0.01
e = 1.602e-19
m_e = 9.10938188e-31
m_p = 1.67262158e-27
k = 1e12 #8.987e9
E_resolution = 500

# computed constants
width = x_max - x_min

E_field = []

def x_dist(x) -> float:
    return 1

def v_dist(v) -> float:
    return np.e**-((25*((v/v_max)-0.3))**2) + np.e**-((25*((v/v_max)+0.3))**2)

In [ ]:
# particle data
name_list: list = []
pos_list:  list = []
vel_list:  list = []
acc_list:  list = []
chg_list:  list = []
mass_list: list = []

lists: list = [name_list, pos_list, vel_list, acc_list, chg_list, mass_list]

E_field: list = [0 for i in range(E_resolution)]

def add_particles(particle: dict, n: int) -> None:
    for i in range(n):
        name_list.append(particle["name"])
        pos_list.append(rand_from_dist(x_dist, x_min, x_max, 100/x_max, True))
        vel_list.append(rand_from_dist(v_dist, v_min, v_max, 100/v_max, True))
        acc_list.append(0)
        chg_list.append(particle["charge"])
        mass_list.append(particle["mass"])

In [ ]:
electron: dict = {
    "name": "Electron",
    "charge": -1*e,
    "mass": m_e
}

ion: dict = {
    "name": "Ion",
    "charge": 1*e,
    "mass": m_p
}

Simulation Functions

In [ ]:
def init() -> None:
    global lists
    for i in range(len(lists)):
        lists[i].clear()
    add_particles(electron, N_e)
    add_particles(ion, N_i)

def E(x) -> float: # electric field
    field_idx = int(E_resolution * (x-x_min) / width)
    if field_idx == E_resolution:
        field_idx -= 1
    if field_idx >= 0 and field_idx < len(E_field):
        return E_field[field_idx]
    print(f"ERROR:\n{x} not in range [{x_min}, {x_max}).\nField Index: {field_idx}")
    raise Exception("Tried to access field cell outside of defined window [x_min, x_max)")


def coulomb_force(j, i) -> float: # coulomb force by particle j on particle i
    L = x_max - x_min
    r_qp = pos_list[i] - pos_list[j]
    r_qp -= L * np.round(r_qp / L)
    dir = r_qp / np.abs(r_qp)
    return k * ((chg_list[i]*chg_list[j])/(r_qp**2))*dir

def upd_accelerations() -> None:
    for i in range(len(name_list)):
        acc_list[i] = E(pos_list[i])

def compute_E_cell(i: int) -> float:
    field = 0
    cell_position = (i * width / E_resolution) + x_min
    N = len(name_list)
    for j in range(N):
        dist = pos_list[j] - cell_position
        field += k * chg_list[j]/(dist**2)
    return field

def upd_E_field() -> None:
    for i in range(len(E_field)):
        E_field[i] = compute_E_cell(i)

def step(frame):
    global ax
    N = len(name_list)
    for i in range(N):
        pos_list[i] += vel_list[i] * dt + 0.5 * acc_list[i] * dt**2
        while pos_list[i] < x_min:
            pos_list[i] += x_max - x_min
        while pos_list[i] > x_max:
            pos_list[i] -= x_max - x_min
    upd_E_field()
    upd_accelerations()
    for i in range(N):
        vel_list[i] += acc_list[i] * dt
    ax.set_title("t = " + str(float(f'{frame*dt:.9f}')))
    print("Current frame: " + str(frame), end='\r')
    view = plt.hist2d(pos_list, vel_list, bins=100, range=[[x_min, x_max], [v_min, v_max]], cmap='hot')
    return view

Creating an Animation

In [ ]:
init()

fig, ax = plt.subplots()
plt.hist2d(pos_list, vel_list, bins=100, range=[[x_min, x_max], [v_min, v_max]], #cmap='hot')
ax.set_xlabel("Position")
ax.set_ylabel("Velocity")
ax.set_box_aspect(1)

In [ ]:
fig, ax = plt.subplots()
ax.set_xlabel("Position")
ax.set_ylabel("Velocity")
ax.set_box_aspect(1)

ani = anim.FuncAnimation(fig=fig, func=step, frames=60, interval=20)
ani.save("first_sim.gif")

In [ ]:
a = [1, 2, 3]
b = [4, 5, 6]
c = [7, 8, 9]

test_list = [a, b, c]

a.append("foo")
test_list[0]

In [ ]:
def test_func(x) -> float:
    return np.cos(x)**2

plt.hist(rfd_array(test_func, 15000, [-3*np.pi/2, 3*np.pi/2]), 100);